# 02 - Summarize and filter a batch

A batch is the collection-level API. This notebook uses the bundled sample to show summary columns and the fact that filtering returns a new batch. Replace the sample path with a glob such as `results/*.log` for a research dataset.

In [1]:
from pathlib import Path

from molop import AutoParser, molopconfig

molopconfig.quiet()
sample_candidates = (
    Path("docs/assets/examples/water_mp2.out"),
    Path("../../assets/examples/water_mp2.out"),
    Path("water_mp2.out"),
)
sample_path = next((path for path in sample_candidates if path.is_file()), None)
if sample_path is None:
    raise FileNotFoundError("Place water_mp2.out beside the notebook or use the bundled example.")
batch = AutoParser(sample_path, n_jobs=1)

print("batch size:", len(batch))
print("files:", batch.file_names)

batch size: 1
files: ['water_mp2.out']


A full summary adds fields that are available in the source. Flattened columns are convenient for CSV and ordinary DataFrame operations.

In [2]:
import pandas as pd
from IPython.display import display

summary = batch.to_summary_df(
    frame=-1,
    brief=False,
    flatten_columns=True,
)
with pd.option_context("display.max_columns", None, "display.max_rows", None, "display.width", 240):
    display(summary)

,DiskStorage.FilePath,DiskStorage.FileFormat,General.Charge,General.Multiplicity,General.CanonicalSMILES,General.NumAtoms,General.FrameID,Calc Parameter.Software,Calc Parameter.Version,Calc Parameter.Method,Calc Parameter.BasisSet,Calc Parameter.Functional,Calc Parameter.Keywords,Environment.SolventModel,Environment.Solvent,Status.IsError,Status.IsNormal,Status.IsTS,Status.IsOptimized,Energy.electronic_energy.hartree,Energy.reference_energy.hartree,Energy.mp2_energy.hartree,Energy.total_energy.hartree
0,/home/tmj/proj/MolOP/docs/assets/examples/wate...,.out,0,1,O,3,0,ORCA,6.0.1,MP2,sto-3g,,MP2 sto-3g,None,None,False,True,False,False,-74.999375,-74.963574,-74.999375,-74.999375


State filters can be chained. They do not mutate the original batch.

In [3]:
normal = batch.filter_state("normal")
optimized = normal.filter_state("opt")
transition_states = normal.filter_state("ts")

print({
    "parsed": len(batch),
    "normal": len(normal),
    "optimized": len(optimized),
    "transition_states": len(transition_states),
})
print("original batch still contains:", len(batch))

{'parsed': 1, 'normal': 1, 'optimized': 0, 'transition_states': 0}
original batch still contains: 1


Use the selected batch for export, or write the summary to a CSV file.

In [4]:
summary.to_csv("summary.csv", index=False)
print("wrote summary.csv")

wrote summary.csv
